**EXPLORATION DES DONNEES INITIALES**

-> Etude des données raw de la table airport_traffic_2025 avant traitement

In [1]:
# 1. Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 2. Chargement des données
url = f"https://docs.google.com/spreadsheets/d/e/2PACX-1vSMsgDXBXJnnqKPblgoqDIxSbjsuwVQqgZBj_HUXYxCDtDbXLzX2AgacNRgb3KrkEwk6K8g4842q-gc/pub?gid=1205565297&single=true&output=csv"
df = pd.read_csv(url)

#NB : lien obtenu via Share - Publish to the Web - Comma-separated..(..)

# 3. Premier coup d'œil
df.shape
df.head()
df.dtypes
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 116020 entries, 0 to 116019
Data columns (total 13 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   YEAR           116020 non-null  int64  
 1   MONTH_NUM      116020 non-null  int64  
 2   MONTH_MON      116020 non-null  object 
 3   FLT_DATE       116020 non-null  object 
 4   APT_ICAO       116020 non-null  object 
 5   APT_NAME       116020 non-null  object 
 6   STATE_NAME     116020 non-null  object 
 7   FLT_DEP_1      116020 non-null  int64  
 8   FLT_ARR_1      116020 non-null  int64  
 9   FLT_TOT_1      116020 non-null  int64  
 10  FLT_DEP_IFR_2  34128 non-null   float64
 11  FLT_ARR_IFR_2  34128 non-null   float64
 12  FLT_TOT_IFR_2  34128 non-null   float64
dtypes: float64(3), int64(5), object(5)
memory usage: 11.5+ MB


**Notes**
#####-> un "mvt" est un "mouvement", un départ (décollage) ou une arrivée (atterissage)
#####-> une escale est définie comme l'espace temps entre un mvt départ et un mvt arrivée où sont effectués toutes les ops au sol; en cas d'absence de mvt départ rattaché, l'avion est considéré comme night ou day-stop cad l'avion ne repart pas le jour J.
#####-> la saisonnalité en aérien est définie par les saisons hiver/été IATA : fin Mars-fin Octobre pour l'été, fin Octobre-fin Mars pour l'hiver, corresponds aux dates de changement d'heure.

**Observations diverses :**

#####-> Colonnes 0,1,2 redondantes avec la colonne 3
#####-> Colonne 3 à convertir en type DATE
#####-> Possibilité d'identifier le ``STATE_NAME`` grâce au préfixe unique* d'``APT_ICAO``
#####-> Etude centrée sur les aéroports commerciaux, colonnes 10,11, 12 permettant d'isoler les aéroports recevant du trafic IFR soit 34128 entrées
#####-> Colonnes 10 à 12 à convertir en INT64 - 1 vol = entier


#####**NB: Exception Serbie/Montenegro cause Ex-Yougoslavie à traiter*

In [2]:
df.duplicated().sum()

np.int64(0)

**Interpretation:** *pas de doublons*

In [7]:
df[['FLT_DEP_IFR_2', 'FLT_ARR_IFR_2', 'FLT_TOT_IFR_2']].describe()

,FLT_DEP_IFR_2,FLT_ARR_IFR_2,FLT_TOT_IFR_2
count,34128.000000,34128.000000,34128.000000
mean,192.939932,192.635314,385.575246
std,168.015658,167.919950,335.881007
min,0.000000,0.000000,1.000000
25%,80.000000,79.000000,159.000000
50%,132.000000,132.000000,263.000000
75%,265.000000,265.000000,531.000000
max,852.000000,850.000000,1696.000000


 **Interpretation:** *34128 entrées, une moyenne de 385 mvts journaliers sur l'année, un max à 1696 mvts sur 1 seule journée soit un mvt toutes les 50,9 sec - pas de différence notable entre les mvts arrivée/départ*

**Roadmap Transformation :**

#####- **Création de la table de faits EU_Traffic_2025**
#####--- ``id`` (primary key)
#####--- ``id_airport`` (foreign key)
#####--- ``date`` (colonne 3 à convertir en type date)
#####--- ``mvt_dep`` (colonne 10 *- avec converstion INT64)*
#####--- ``mvt_arr`` (colonne 11 *- avec converstion INT64)*
#####--- ``mvt_total`` (10+11)
#####--- ``night_stops`` (11-10)

#####- **Création de la table de dimension EU_Airports**
#####--- ``id_airport`` (primary key)
#####--- ``ICAO_code`` (reprise colonne 4)
#####--- ``full_name`` (reprise colonne 5)
#####--- ``id_state`` (foreign key)

#####- **Création de la table de dimension EU_ICAO_by_Countries**
#####--- ``id_state`` (primary key)
#####--- ``ICAO_code`` (reprise colonne 4)
#####--- ``state`` (colonne calculée à partir d'une db gsheet - isole le préfixe à 2 digits et renvoie le résultat correspondant)

Piste de développpement : lié les aéroports étudiés à leur données pistes pour extraire des coefficients d'utilisation - utilisation des db OurAirports pour base de travail, nettoyage et JOIN ;





**Exploration des DB OurAiports**

*Téléchargement des .csv puis important dans GSheets dédiés*

**Airports**

In [9]:
# Chargement des données
url = f"https://docs.google.com/spreadsheets/d/e/2PACX-1vRDYi0IWKuk00TidJGelabDx45LP_IeG2m7KCD90ZlXEh7Xhg_1ltY9MA0lVPzmAcuZRZHYMZaEm3GE/pub?gid=1319254416&single=true&output=csv"
df_airports = pd.read_csv(url)

# Premier coup d'œil
df_airports.shape
df_airports.head()
df_airports.dtypes
df_airports.info()
df_airports.duplicated().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 85846 entries, 0 to 85845
Data columns (total 24 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id                 85846 non-null  int64  
 1   ident              85846 non-null  object 
 2   type               85846 non-null  object 
 3   name               85846 non-null  object 
 4   latitude_deg       85846 non-null  float64
 5   longitude_deg      85846 non-null  float64
 6   elevation_ft       70971 non-null  float64
 7   continent          46136 non-null  object 
 8   country_name       85846 non-null  object 
 9   iso_country        85543 non-null  object 
 10  region_name        85846 non-null  object 
 11  iso_region         85846 non-null  object 
 12  local_region       85829 non-null  object 
 13  municipality       81131 non-null  object 
 14  scheduled_service  85846 non-null  int64  
 15  gps_code           44394 non-null  object 
 16  icao_code          104

np.int64(0)

**Runways**

In [10]:
# Chargement des données
url = f"https://docs.google.com/spreadsheets/d/e/2PACX-1vSBfzVs1ntxj61r6HoyUmFpDka9SKdYIxOAwe_w8f7uDtUxsDg-sc1cOmW7tKBH7Anq_7xfU8rEVJQE/pub?gid=2074982456&single=true&output=csv"
df_runways = pd.read_csv(url)

# Premier coup d'œil
df_runways.shape
df_runways.head()
df_runways.dtypes
df_runways.info()
df_runways.duplicated().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48146 entries, 0 to 48145
Data columns (total 20 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   id                         48146 non-null  int64  
 1   airport_ref                48146 non-null  int64  
 2   airport_ident              48146 non-null  object 
 3   length_ft                  47855 non-null  float64
 4   width_ft                   45131 non-null  float64
 5   surface                    47640 non-null  object 
 6   lighted                    48146 non-null  int64  
 7   closed                     48146 non-null  int64  
 8   le_ident                   47903 non-null  object 
 9   le_latitude_deg            15670 non-null  float64
 10  le_longitude_deg           15657 non-null  float64
 11  le_elevation_ft            13311 non-null  float64
 12  le_heading_degT            15083 non-null  float64
 13  le_displaced_threshold_ft  3067 non-null   flo

np.int64(0)

**Observations:**
#####-> ne seront retenu que les données des aéroports eu dans notre table de faits initiale (via un df.merge ?) pour création d'une table de dimension Airports plus fournie ; la table ourAirports - GEN n'apporte rien en l'état actuel

#####-> ajout des infos : ``surface``, ``lenght`` ( la plus élevée du nbre de runways/airport) et ajout d'une colonne nombre de runways by airport calculée

**Nettoyage de la colonne Surface**

In [11]:
df_runways['surface'].unique()

array(['ASPH-G', 'GRVL', 'TURF', 'GVL', 'Turf', 'GRAVEL', 'ASPH',
       'TURF-F', 'MATS', 'CONC', 'TURF-G', 'CON', 'Turf/Dirt', 'TURF-P',
       'GRAVEL-F', 'ASPH-TRTD', 'TURF-GRVL', 'WATER', 'ASPH-TURF', 'DIRT',
       'CONC-G', 'DIRT-P', 'DIRT-TURF-G', 'PSP', 'CONC-TURF', nan, 'Dirt',
       'DIRT-G', 'TURF-DIRT', 'ASP', 'GRVL-DIRT', 'DIRT-F', 'GRVL-G',
       'ASPH-CONC-G', 'WATER-E', 'ASP-G', 'CONC-E', 'TURF-GRVL-F',
       'Water', 'ROOF-TOP', 'ASPH-P', 'Deck', 'ASPH-F', 'ASPH-E',
       'Concrete/Turf', 'Concrete/Dirt', 'ASPH-DIRT', 'ASPH-TRTD-P',
       'TREATED', 'SAND', 'Roof/Top', 'GRVL-TRTD-P', 'TURF-DIRT-F',
       'TURF-GRVL-P', 'SOD', 'Torf', 'Asph/Conc', 'TURF-E', 'WOOD',
       'ALUM', 'ASPH-TURF-P', 'ASPH-CONC', 'GRASS', 'GRVL-P', 'GRAVEL-P',
       'TURF-DIRT-P', 'TRTD-DIRT', 'GRVL-TRTD-F', 'Concrete Rooftop',
       'WAT', 'Rooftop', 'ASPH-GRVL', 'WATER-G', 'GRVL-F', 'CONC-F',
       'GRASS / SOD', 'TURF-GRVL-G', 'DIRT-TURF', 'GRS', 'GRAVEL-G',
       'Concrete/Grav

In [12]:
df_runways['surface'].value_counts().head(15)

,count
surface,
ASP,11369
TURF,7489
CON,3648
CONC,3101
GRS,2242
ASPH,1678
GRE,1537
Turf,1305
GVL,1067


#####-> valeurs à rassembler sous un ensemble moins large de caractéristiques, il faudra créé de nouvelles catégories.

**NETTOYAGE DES DATASETS**

Roadmap:
#####1- Conversion de la col ``DATE`` en type date *iso object*
#####2- Remplacer les valeurs manquantes par 0 (``FLT_IFR``)
#####3- Conversion des col ``FLT_IFR`` en Int64 *iso Float64*
#####4- Suppression des données hors trafic commercial (filtrage via FLT_IFR=!0)
#####5- Suppression des aéroports non utilisés dans l'étude dans la db Runways
#####6- Nettoyage de la col ``surface``

####**1- Conversion de la col ``DATE`` en type date *iso object***

In [14]:
df['FLT_DATE'].isna().sum()

np.int64(0)

In [16]:
df['FLT_DATE'] = pd.to_datetime(df['FLT_DATE'])

####**2- Remplacer les valeurs manquantes par 0 (``FLT_IFR``)**


In [20]:
df[['FLT_DEP_IFR_2', 'FLT_ARR_IFR_2', 'FLT_TOT_IFR_2']]=df[['FLT_DEP_IFR_2', 'FLT_ARR_IFR_2', 'FLT_TOT_IFR_2']].fillna(0)

####**3- Conversion des col ``FLT_IFR`` en Int64 *iso Float64***


In [21]:
df[['FLT_DEP_IFR_2', 'FLT_ARR_IFR_2', 'FLT_TOT_IFR_2']].astype(int)

,FLT_DEP_IFR_2,FLT_ARR_IFR_2,FLT_TOT_IFR_2
0,0,0,0
1,0,0,0
2,0,0,0
3,0,0,0
4,0,0,0
...,...,...,...
116015,0,0,0
116016,0,0,0
116017,0,0,0
116018,137,137,274


####**4- Suppression des données hors trafic commercial (filtrage via ``FLT_IFR``=!0)***

In [22]:
df = df[~((df['FLT_DEP_IFR_2'] == 0) & (df['FLT_ARR_IFR_2'] == 0))]

In [23]:
print(df.shape)

(34128, 13)


####**5- Suppression des aéroports non utilisés dans l'étude dans la db Runways***

In [24]:
df_runways = df_runways[df_runways['airport_ident'].isin(df['APT_ICAO'])]

In [26]:
print(df_runways['airport_ident'].head())

11098    BIKF
11099    BIKF
15641    EBBR
15642    EBBR
15643    EBBR
Name: airport_ident, dtype: object


####**6- Nettoyage de la col ``surface``**

*seules les surfaces en concrete ou asphaltes nous interessent, cela va simplifier le nettoyage en trois grandes catégories uniquement, ASP, CON et OTH (OTH=others)*




In [28]:
df_runways.loc[df_runways['surface'].str.contains('asp', case=False, na=False), 'surface'] = 'ASP'

In [29]:
df_runways.loc[df_runways['surface'].str.contains('con', case=False, na=False), 'surface'] = 'CON'

In [30]:
df_runways.loc[~(df_runways['surface'].str.contains('asp', case=False, na=False)|df_runways['surface'].str.contains('con', case=False, na=False)),'surface'] = 'OTH'

In [31]:
print(df_runways['surface'].head())

11098    ASP
11099    ASP
15641    ASP
15642    ASP
15643    ASP
Name: surface, dtype: object


**Les données sont nettoyées, place à la transformation.**